In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from scipy.interpolate import interp1d

C:\ProgramData\Anaconda3\lib\site-packages\scipy\__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


In [ ]:
#### INPUT mean particle radius of CAM material ####
rp = 0.0002  # mean CAM particle radius in cm

#### INPUT appropriate Arbin protocol step numbers ####
charge_step = 8
pause_step = 7

#### INPUT filepaths under appropriate sample name, can input as many cells / samples as you'd like####
sample_files = {
    "sample-1": [
        r,
    ],
    "sample-2": [
        r,
    ],
    "sample-3": [
        r,
    ],
}

#### plotting setup ####
fig, axes = plt.subplots(2, 1, figsize=(15, 12), sharex=True)
colors = plt.cm.tab20.colors  # Up to 20 unique colors

summary_data = []  # to store max vals and their corresponding voltages

#### process each sample ####
for i, (sample_name, file_list) in enumerate(sample_files.items()):
    color = colors[i % len(colors)]  # assign unique color per sample
    valid_voltage_ranges = []  # reset for each sample
    interp_diff = []  # reset for each sample

    for file_path in file_list:
        df = pd.read_excel(file_path, sheet_name=1)
        D_Li = []  # store charge diff coef data
        file_label = file_path.split('\\')[-1][:-5]  # label for plotting

        ICI_steps = list(range(10, df['Cycle Index'].max() - 3))

        for step in ICI_steps:
            #### Charge analysis ####
            charge_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == charge_step)].copy()
            pause_data = df[(df['Cycle Index'] == step) & (df['Step Index'] == pause_step)].copy()
            next_pause_data = df[(df['Cycle Index'] == step + 1) & (df['Step Index'] == pause_step)].copy()

            if not (charge_data.empty or pause_data.empty or next_pause_data.empty):
                try:
                    tau = charge_data['Test Time (s)'].iloc[-1] - charge_data['Test Time (s)'].iloc[0]  # charge duration
                    t0 = pause_data['Test Time (s)'].iloc[0]
                    mask = (pause_data['Test Time (s)'] - t0 >= 0.1) & (pause_data['Test Time (s)'] - t0 <= 5)  # early pause data
                    dEt_data = pause_data[mask]

                    if not dEt_data.empty:
                        sqrt_t = np.sqrt(dEt_data['Test Time (s)'] - t0)
                        voltage = dEt_data['Voltage (V)']
                        slope, _, _, _, _ = linregress(sqrt_t, voltage)
                        dEt = abs(slope)  # slope of E with sqrt time - linear
                        dEs = abs(next_pause_data['Voltage (V)'].iloc[0] - pause_data['Voltage (V)'].iloc[0])
                        Eeq_vals = pause_data['Voltage (V)'].iloc[0]

                        if dEt != 0 and tau != 0:
                            D = (4 / (np.pi * 9)) * ((rp / tau) ** 2) * ((dEs / dEt) ** 2)
                            D_Li_c.append({'Step': step, 'Diff_coef (cm^2/s)': D, 'Eeq (V)': Eeq_vals})
                except Exception as e:
                    print(f"{file_label} CHARGE: Error at step {step} - {e}")

        results_df = pd.DataFrame(D_Li)

        if not results_df.empty:
            axes[0].scatter(results_df['Eeq (V)'], results_df['Diff_coef (cm^2/s)'],
                            label=f"{sample_name}", color=color, s=30, edgecolors='black', alpha=0.6)

            charge_data_sorted = results_df.sort_values('Eeq (V)')
            vmin = charge_data_sorted['Eeq (V)'].min()
            vmax = charge_data_sorted['Eeq (V)'].max()
            valid_voltage_ranges.append((vmin, vmax))

            interp_chg_diff.append((charge_data_sorted['Eeq (V)'], charge_data_sorted['Diff_coef (cm^2/s)']))

    #### voltage grid for interpolation ####
    if valid_voltage_ranges:
        vmin_all = max(v[0] for v in valid_voltage_ranges)
        vmax_all = min(v[1] for v in valid_voltage_ranges)
        voltage_grid = np.linspace(vmin_all, vmax_all, 1000)

        #### interpolate diffusion values ####
        interp_matrix = []
        for v_vals, d_vals in interp_diff:
            interp_func = interp1d(v_vals, d_vals, kind='linear', bounds_error=False, fill_value=np.nan)
            interp_matrix.append(interp_func(voltage_grid))

        matrix = np.array(interp_matrix)

        #### mean & std dev ####
        mean_diff = np.nanmean(matrix, axis=0)
        std_diff = np.nanstd(matrix, axis=0)

        #### plot results ####
        axes[1].plot(voltage_grid, mean_diff, color=color, linewidth=2.5, label=f"{sample_name} Mean")
        axes[1].fill_between(voltage_grid,
                             mean_diff - std_diff,
                             mean_diff + std_diff,
                             color=color, alpha=0.3)

        #### get max diffusion coefficient ####
        max_idx = np.nanargmax(mean_diff)
        max_voltage = voltage_grid[max_idx]
        max_diff_val = mean_diff[max_idx]
        max_std_val = std_diff[max_idx]

        summary_data.append({
            "Sample": sample_name,
            "Voltage (V)": round(max_voltage, 3),
            "Mean Diff Coef (cm^2/s)": max_diff_val,
            "Std Dev": max_std_val,
            "Diff Coef (cm^2/s)": f"{max_diff_val:.2e} ± {max_std_val:.2e}"
            })

#### plot formatting ####
axes[0].set_title('ICI calculated Li diff coefficients', fontsize=18, fontweight='bold')
axes[0].set_ylabel('Li Diff Coef (cm$^2$/s)', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Cell Potential (V)')
axes[0].set_ylim(-0.1e-11, 2.5e-11)
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].set_title("Mean and Standard Deviation", fontsize=18, fontweight='bold')
axes[1].set_ylabel('Li Diff Coef (cm$^2$/s)', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Cell Potential (V)', fontsize=16, fontweight='bold')
axes[1].set_ylim(-0.1e-11, 2.5e-11)
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=15)

plt.tight_layout()
fig.subplots_adjust(top=0.9)
fig.canvas.draw()

### INPUT name of study to save the figure with the correct name ###
plt.savefig('study-1', dpi=300)
plt.show()

#### convert to dataframe ####
summary_df = pd.DataFrame(summary_data)
print("\n=== Peak Diffusion Summary ===\n")
print(summary_df[['Sample', 'Voltage (V)', 'Diff Coef (cm^2/s)']])

### INPUT name of study to save data summary with the correct name ###
summary_df.to_csv("diffusion_summary_study-1.csv", index=False)